# Goals of this analysis
The graphs we are trying to generate
1. Initial distribution of computing time
2. Distribution of computing time after splitting on user device computation time out
3. Distribution of computing time after removing running proof generating in parallel 

We also need to see how long each of these steps take for 5 million votes. And we need to see how long the final time is when we start to speed up message processing.

# How to get the data
Setup the three files to collect the runs you want.
```
__custom_benchmarks__/initial_proof_gen_work.ts
__custom_benchmarks__/split_out_client_side_work.ts
__custom_benchmarks__/parallelized_proof_gen.ts
```

Run them with the following commands
```
yarn run proof_bench_1
yarn run proof_bench_2
yarn run proof_bench_3
```

Move the generated json files to a separate folder for analysis.

Enter the file names below

In [9]:
PROOF_BENCH_1_OUTPUT_FILE = "Profile_initial_proof_gen_work.json"
PROOF_BENCH_2_OUTPUT_FILE = "Profile_split_out_client_side_work.json"
PROOF_BENCH_3_OUTPUT_FILE = "Profile_parallelized_proof_gen.json"

Plotting bench 1 and 2 for the report

In [10]:
#!/usr/bin/env python3
import json
import sys
from pathlib import Path
from typing import Dict, List, Iterable, Tuple

import matplotlib.pyplot as plt
from matplotlib.patches import Patch


NICE_COLORS = [
    "#4C72B0",  # muted blue
    "#55A868",  # soft green
    "#C44E52",  # brick red
    "#8172B2",  # purple-gray
    "#CCB974",  # mustard
    "#64B5CD",  # teal
    "#8C8C8C",  # neutral gray
]

# ------------- I/O -------------

def load_results(path: Path):
    """Load and return the parsed JSON dict."""
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


# ------------- Grouping helpers -------------

def _normalize_sections(sections_or_fields: Dict, sample_record: dict) -> Tuple[List[List[str]], List[str]]:
    """
    Accepts either:
      1) New grouped style:
         {
           "Compute (CPU+GPU)": ["cpuTime", "gpuTime"],
           "I/O": ["diskTime", "netTime"],
           "Overhead": "overheadTime",
         }
      2) Back-compat (old 'fields' style: key->label, no grouping):
         {
           "cpuTime": "CPU",
           "gpuTime": "GPU",
           ...
         }

    Returns:
        groups: list of lists of raw keys (each inner list is summed into one series)
        labels: list of legend labels (same order as groups)
    """
    # Detect style: if any value is a list or a single known key string (not a label),
    # we assume the *new grouped* style (title -> raw key(s)).
    # Otherwise we assume old style (raw key -> label).
    # Heuristic: check whether dict values are list/str AND those strings look like keys in record.
    record_keys = set(k for k in sample_record.keys() if k != "data")

    def looks_like_key(v: str) -> bool:
        return v in record_keys

    is_grouped = any(
        isinstance(v, (list, tuple)) or (isinstance(v, str) and looks_like_key(v))
        for v in sections_or_fields.values()
    )

    groups: List[List[str]] = []
    labels: List[str] = []

    if is_grouped:
        # New style: title -> key(s)
        for title, keys in sections_or_fields.items():
            if isinstance(keys, str):
                keys = [keys]
            keys = list(keys)
            # Validate keys exist; warn if missing
            missing = [k for k in keys if k not in record_keys]
            if missing:
                print(f"Warning: Missing keys in data for group '{title}': {missing}")
            # Keep only existing keys to avoid KeyError
            keys = [k for k in keys if k in record_keys]
            if not keys:
                # Skip empty groups entirely
                continue
            groups.append(keys)
            labels.append(title)
    else:
        # Old style: rawKey -> label
        for raw_key, label in sections_or_fields.items():
            if raw_key not in record_keys:
                print(f"Warning: Missing key in data: '{raw_key}'")
                continue
            groups.append([raw_key])
            labels.append(label)

    if not groups:
        raise ValueError("No valid groups/fields found after normalization.")
    return groups, labels


def _sum_series_for_group(records: Iterable[dict], keys: List[str]) -> List[float]:
    """Sum multiple fields for each record to produce one series."""
    out = []
    for rec in records:
        total = 0.0
        for k in keys:
            total += rec.get(k, 0.0)
        out.append(total)
    return out


# ------------- Data shaping -------------

def _flatten_results(results: dict):
    """
    Returns:
        flat: list of (numUsers, record) sorted by numUsers asc
    """
    flat = [(rec["data"]["numUsers"], rec) for rec in results.values()]
    flat.sort(key=lambda t: t[0])
    return flat


def extract_series_grouped(results: dict, groups: List[List[str]]):
    """
    Return:
        x      : list[int]    – number of users
        stacks : list[list]   – each inner list holds the summed series for a group
    """
    flat = _flatten_results(results)
    x = [num for num, _ in flat]
    records = [rec for _, rec in flat]
    stacks = [_sum_series_for_group(records, keys) for keys in groups]
    return x, stacks


# ------------- Plotting -------------

def plot(filename, title, sections_or_fields: Dict):
    """
    Draw a stacked area chart.

    Args:
        filename: path to JSON results
        title: chart title
        sections_or_fields:
           - New grouped style: {"Legend Title": ["rawKey1", "rawKey2"], ...}
           - Back-compat:       {"rawKey": "Legend Title", ...}
    """
    json_path = Path(filename)
    if not json_path.exists():
        print(f"Error: file {json_path} does not exist.")
        sys.exit(1)

    results = load_results(json_path)

    # Use the first record to infer key style
    sample_record = next(iter(results.values()))
    groups, labels = _normalize_sections(sections_or_fields, sample_record)

    x, stacks = extract_series_grouped(results, groups)

    # plotting
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.stackplot(x, stacks, labels=labels, colors=NICE_COLORS[: len(labels)])
    ax.set_title(title)
    ax.set_xlabel("Number of users")
    ax.set_ylabel("Time (seconds)")
    ax.legend(loc="upper left")

    ax.tick_params(axis="x", rotation=45)

    out_png = json_path.with_name(json_path.stem + "_stacked_area.png")
    plt.tight_layout()
    plt.savefig(out_png, dpi=600)
    print(f"Saved plot to {out_png.resolve()}")
    plt.show()


def find_record_by_users(results: dict, num_users: int):
    """Return the record with the specified numUsers value, or None if not found."""
    for rec in results.values():
        if rec["data"]["numUsers"] == num_users:
            return rec
    return None


def _labels_and_values_for(filename: str, sections_or_fields: Dict, num_users: int):
    """Helper: returns (labels, values) for a given file and num_users."""
    json_path = Path(filename)
    if not json_path.exists():
        print(f"Error: file {json_path} does not exist.")
        sys.exit(1)

    results = load_results(json_path)
    record = find_record_by_users(results, num_users)
    if record is None:
        print(f"Error: No record found for numUsers={num_users}.")
        sys.exit(1)

    groups, labels = _normalize_sections(sections_or_fields, record)
    values = []
    for keys in groups:
        total = 0.0
        for k in keys:
            total += record.get(k, 0.0)
        values.append(total)
    return labels, values


def plot_pies_side_by_side(
    left,   # (filename, sections_or_fields, num_users, title)
    right,  # (filename, sections_or_fields, num_users, title)
    figure_title: str | None = None,
):
    """
    Draw two pie charts next to each other with a shared legend.
    Labels subplots as (a) and (b), and keeps legend outside the pies.
    """
    (file1, sect1, users1, title1) = left
    (file2, sect2, users2, title2) = right

    labels1, values1 = _labels_and_values_for(file1, sect1, users1)
    labels2, values2 = _labels_and_values_for(file2, sect2, users2)

    # Build a stable union of labels so colors are consistent across both pies.
    unified_labels: List[str] = []
    for L in (labels1, labels2):
        for lab in L:
            if lab not in unified_labels:
                unified_labels.append(lab)

    # Map each label to a color (cycled if needed).
    color_map = {lab: NICE_COLORS[i % len(NICE_COLORS)] for i, lab in enumerate(unified_labels)}

    def autopct_func(pct):
        return f"{pct:.1f}%" if pct >= 4 else ""

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

    wedges1, _, autotexts1 = ax1.pie(
        values1,
        colors=[color_map[l] for l in labels1],
        startangle=90,
        autopct=autopct_func,
        wedgeprops=dict(linewidth=1, edgecolor="white"),
    )
    ax1.set_title(title1)
    ax1.axis("equal")
    ax1.text(-0.05, 1.05, "(a)", transform=ax1.transAxes, fontsize=12, fontweight="bold", va="bottom")

    wedges2, _, autotexts2 = ax2.pie(
        values2,
        colors=[color_map[l] for l in labels2],
        startangle=90,
        autopct=autopct_func,
        wedgeprops=dict(linewidth=1, edgecolor="white"),
    )
    ax2.set_title(title2)
    ax2.axis("equal")
    ax2.text(-0.05, 1.05, "(b)", transform=ax2.transAxes, fontsize=12, fontweight="bold", va="bottom")

    # Single shared legend below the charts (no overlap).
    legend_handles = [Patch(facecolor=color_map[l], edgecolor="white") for l in unified_labels]
    fig.legend(
        legend_handles,
        unified_labels,
        title="Components",
        loc="lower center",
        ncol=min(4, len(unified_labels)),
        frameon=False,
    )

    if figure_title:
        fig.suptitle(figure_title, y=0.98)

    # Leave room at the bottom for the legend and tighten.
    fig.subplots_adjust(bottom=0.20)
    plt.tight_layout(rect=[0, 0.08, 1, 0.97])

    # Save next to the first file by default.
    out_png = Path(file1).with_name(Path(file1).stem + "_two_pies.svg")
    plt.savefig(out_png, dpi=300, transparent=True, format="svg")
    print(f"Saved combined pies to {out_png.resolve()}")
    plt.show()

In [ ]:
PROOF_BENCH_2_TITLE = "Time consumption by process"
PROOF_BENCH_2_FIELDS = {
    "Setup and Join Poll" : ["SIGN_UP", "UPDATE_POLL", "JOIN_POLL"],
    "Loading Votes": ["TOTAL_VALID_VOTES"],
    "Message Processing": ["PROCESS_MESSAGES"],
    "Proof Generation": ["TALLY_PROOFS"],
}
plot(PROOF_BENCH_2_OUTPUT_FILE, PROOF_BENCH_2_TITLE, PROOF_BENCH_2_FIELDS)

PARALLEL_BENCH_FIELDS = PROOF_BENCH_2_FIELDS.copy()
del PARALLEL_BENCH_FIELDS["Proof Generation"]
PARALLEL_BENCH_FIELDS["Proof Generation"] = ["SAVE_CIRCUIT_INPUTS"]

plot_pies_side_by_side(
    left=(
        PROOF_BENCH_2_OUTPUT_FILE,
        PROOF_BENCH_2_FIELDS,
        6400,
        "Percentage time by process",
    ),
    right=(
        PROOF_BENCH_3_OUTPUT_FILE,
        PARALLEL_BENCH_FIELDS,
        6400,
        "Percentage time by process (parallelized)",
    ),
    figure_title=None,  # or set a common title string if you want
)


# Now extrapolate to 5 million votes

In [ ]:
NUM_VOTES_TO_USE_AS_INPUT = 50

def print_seconds(seconds ):
    # if seconds > 86400:
        print(f"{seconds / 86400:.2f} days")
    # if seconds > 604800:
        print(f"Or about {seconds / 604800:.2f} weeks")

def extrapolate_to_n_votes(filename, fields, n, modifiers=dict()):
    results = load_results(Path(filename))
    x, stacks = extract_series(results, list(fields.keys()))

    for i, num in enumerate(x):
        if num == NUM_VOTES_TO_USE_AS_INPUT:
            scale_factor = n / float(NUM_VOTES_TO_USE_AS_INPUT)
            extrapolated_stack = [v[i] * scale_factor for v in stacks] 

            for j, key in enumerate(fields.keys()):
                if key in modifiers:
                    extrapolated_stack[j] = modifiers[key] * extrapolated_stack[j]
            return n, extrapolated_stack

    raise ValueError(f"Could not find data for {NUM_VOTES_TO_USE_AS_INPUT} votes in {filename}")


EXTRAPOLATE_FIELDS = PROOF_BENCH_2_FIELDS
EXTRAPOLATE_FIELDS["MP_PROOFS"] = "Message processing proofs"

n, extrapolated_stack = extrapolate_to_n_votes(PROOF_BENCH_2_OUTPUT_FILE, EXTRAPOLATE_FIELDS, 5_000_000)
print(f"Non Parallel Time for {n} votes:")
non_parallel_seconds = sum(extrapolated_stack)
print_seconds(non_parallel_seconds)


PARALLEL_EXTRAPOLATE_FIELDS = EXTRAPOLATE_FIELDS.copy()
del PARALLEL_EXTRAPOLATE_FIELDS["TALLY_PROOFS"]
PARALLEL_EXTRAPOLATE_FIELDS["SAVE_CIRCUIT_INPUTS"] = "Prepare proof circuit inputs"


n, parallel_extrapolated_stack = extrapolate_to_n_votes(PROOF_BENCH_3_OUTPUT_FILE, PARALLEL_EXTRAPOLATE_FIELDS, 5_000_000)
print(f"\nParallel Time for {n} votes:")
parallel_seconds = sum(parallel_extrapolated_stack)
print_seconds(parallel_seconds)


def process_messages_speedup(speedup: float):
    modifiers = {"PROCESS_MESSAGES": 1/speedup} # assume 1.5x speed up due to parallelism
    n, modified_parallel_extrapolated_stack = extrapolate_to_n_votes(PROOF_BENCH_3_OUTPUT_FILE, PARALLEL_EXTRAPOLATE_FIELDS, 5_000_000, modifiers=modifiers)
    print(f"\n{speedup}x message processing time for {n} votes:")
    parallel_seconds = sum(modified_parallel_extrapolated_stack)
    print_seconds(parallel_seconds)
    return parallel_seconds

process_messages_speedup(1.5)
process_messages_speedup(2.0)
process_messages_speedup(3.0)
process_messages_speedup(4.0)
process_messages_speedup(5.0)

NameError: name 'extract_series' is not defined